In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Franz Wagner,Over,24.5,-137,2025-11-15,2025-11-14T18:43:22Z
1,Underdog,player_points,Franz Wagner,Under,24.5,-137,2025-11-15,2025-11-14T18:43:22Z
2,Underdog,player_points,Desmond Bane,Over,19.5,-137,2025-11-15,2025-11-14T18:43:22Z
3,Underdog,player_points,Desmond Bane,Under,19.5,-137,2025-11-15,2025-11-14T18:43:22Z
4,Underdog,player_points,Nicolas Claxton,Over,13.5,-137,2025-11-15,2025-11-14T18:43:22Z


### Update projected starting lineups

In [5]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Applications/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [6]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=10, 
                             variance_inflation=1.1, 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(10)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 132 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Luka Doncic,Bovada,28.5,20.35,Under,205,1,15.58,155.8,0.760,High
1,Luka Doncic,Bovada,29.5,20.35,Under,170,1,13.47,134.7,0.793,High
2,Luka Doncic,Bovada,30.5,20.35,Under,145,1,11.77,117.7,0.812,High
3,Luka Doncic,Bovada,31.5,20.35,Under,120,1,10.02,100.2,0.835,High
4,Luka Doncic,BetRivers,32.5,20.35,Under,114,1,9.94,99.4,0.872,High


## Top EVs for 2 leg bets

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 90 players...
Processing 83 players with valid predictions...
Generated 3050 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Deni Avdija,Luka Dončić,23.5,33.5,over,under,1,10.01,0.500,High,High
1,Luka Dončić,Stephen Curry,33.5,25.5,under,over,1,8.42,0.421,High,High
2,Shaedon Sharpe,Luka Dončić,19.5,33.5,over,under,1,7.80,0.390,High,High
3,Luka Dončić,Brandon Williams,33.5,13.5,under,over,1,7.75,0.388,High,High
4,Kon Knueppel,Luka Dončić,15.5,33.5,over,under,1,7.52,0.376,High,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 116 players...
Processing 106 players with valid predictions...
Generated 4975 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Deni Avdija,Luka Dončić,23.5,33.5,over,under,1,10.28,0.514,High,High
1,Jordan Clarkson,Luka Dončić,11.5,33.5,under,under,1,8.29,0.415,Med,High
2,Luka Dončić,Stephen Curry,33.5,25.5,under,over,1,8.21,0.411,High,High
3,Luka Dončić,Brandon Williams,33.5,13.5,under,over,1,8.07,0.404,High,High
4,Norman Powell,Luka Dončić,22.5,33.5,over,under,1,7.68,0.384,High,High


## 3 leg parlay

### Underdog picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1, 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 90 players...
Processing 83 players with valid predictions...
Generated 90966 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Deni Avdija,Shaedon Sharpe,Luka Dončić,23.5,19.5,33.5,over,over,under,0,16.19,0.324,High,High,High
1,Deni Avdija,Luka Dončić,Stephen Curry,23.5,33.5,25.5,over,under,over,0,15.66,0.313,High,High,High
2,Miles McBride,Deni Avdija,Luka Dončić,12.5,23.5,33.5,under,over,under,0,15.36,0.307,Med,High,High
3,Deni Avdija,Luka Dončić,Brandon Williams,23.5,33.5,13.5,over,under,over,0,15.17,0.303,High,High,High
4,Jalen Wilson,Deni Avdija,Luka Dončić,5.5,23.5,33.5,over,over,under,0,15.08,0.302,Med,High,High


### Prizepicks picks

In [12]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=100, 
                     variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 116 players...
Processing 106 players with valid predictions...
Generated 190871 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Miles McBride,Deni Avdija,Luka Dončić,12.5,23.5,33.5,under,over,under,0,19.18,0.384,Med,High,High
1,Miles McBride,Luka Dončić,Jonathan Kuminga,12.5,33.5,10.5,under,under,over,0,17.13,0.343,Med,High,High
2,Deni Avdija,Luka Dončić,Jonathan Kuminga,23.5,33.5,10.5,over,under,over,0,16.90,0.338,High,High,High
3,Deni Avdija,Shaedon Sharpe,Luka Dončić,23.5,19.5,33.5,over,over,under,0,16.19,0.324,High,High,High
4,Miles McBride,Luka Dončić,Stephen Curry,12.5,33.5,25.5,under,under,over,0,16.02,0.320,Med,High,High
